# TP 2 : Retrieval Avance

## Objectif

Implementer et comparer differentes techniques de retrieval avancees pour identifier la meilleure approche selon le type de requete.

## Enonce

### Contexte

Vous construisez un systeme de recherche pour une base de connaissances d'entreprise contenant 100+ documents. Vous devez choisir la meilleure strategie de retrieval.

### Taches

1. **Creer une base vectorielle** avec 100+ documents (fournis)
2. **Implementer 3 types de recherche** :
   - Similarity Search classique
   - MMR avec lambda=0.3, 0.5, 0.7
   - Hybrid Search (BM25 + Vectoriel) avec differents weights
3. **Tester sur 5 questions variees** (techniques, conceptuelles, etc.)
4. **Analyser la diversite** : categories uniques, auteurs uniques
5. **Mesurer les temps de reponse**

### Questions de test

1. "Comment optimiser les performances d'une base de donnees PostgreSQL ?"
2. "Quelles sont les techniques de regularisation en deep learning ?"
3. "Docker Kubernetes orchestration conteneurs"
4. "Expliquer le reranking dans un systeme RAG"
5. "API REST OAuth authentication security"


## Votre Implementation

In [1]:
# Imports
import warnings

warnings.filterwarnings("ignore")

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

import numpy as np
import pandas as pd
import time

# TODO: Votre code ici


### TODO 1: Creer un dataset de documents

Creez au moins 100 documents couvrant differents sujets (tech, business, RH, etc.)

In [2]:
# Creer les documents
documents = [
    # Documents Database (15)
    Document(
        page_content="Pour optimiser les performances d'une base de données PostgreSQL, plusieurs techniques sont essentielles. Les index B-tree doivent être créés sur les colonnes fréquemment utilisées dans les clauses WHERE et JOIN. Utilisez EXPLAIN ANALYZE pour identifier les requêtes lentes. Le vacuum régulier empêche le table bloat. La configuration des paramètres shared_buffers, work_mem et effective_cache_size selon la RAM disponible améliore drastiquement les performances.",
        metadata={"category": "database", "author": "Marie Dubois", "year": 2024},
    ),
    Document(
        page_content="En production, PostgreSQL nécessite une configuration minutieuse. Le paramètre max_connections doit être ajusté selon la charge. Le checkpoint_completion_target à 0.9 lisse les écritures disque. Les logs doivent être configurés avec log_min_duration_statement pour tracer les requêtes lentes. La réplication streaming assure la haute disponibilité.",
        metadata={"category": "database", "author": "Jean Martin", "year": 2024},
    ),
    Document(
        page_content="MongoDB utilise une architecture distribuée avec le sharding pour la scalabilité horizontale. Le shard key détermine la distribution des données entre shards. Un bon shard key doit avoir une cardinalité élevée et une distribution uniforme. Les config servers stockent les métadonnées du cluster.",
        metadata={"category": "database", "author": "Sophie Chen", "year": 2023},
    ),
    Document(
        page_content="Redis est un store clé-valeur en mémoire idéal pour le caching. Les stratégies d'éviction comme LRU (Least Recently Used) gèrent la mémoire limitée. Les structures de données comme hashes, sets et sorted sets optimisent différents use cases. La persistence avec RDB snapshots et AOF logs équilibre performance et durabilité.",
        metadata={"category": "database", "author": "Pierre Lefebvre", "year": 2024},
    ),
    Document(
        page_content="Le protocole Two-Phase Commit (2PC) garantit la cohérence des transactions distribuées. Dans la phase de préparation, le coordinateur demande à tous les participants s'ils peuvent commiter. Si tous répondent oui, la phase de commit est exécutée.",
        metadata={"category": "database", "author": "Marie Dubois", "year": 2023},
    ),
    Document(
        page_content="Le théorème CAP stipule qu'un système distribué ne peut garantir simultanément Consistency, Availability et Partition tolerance. En pratique, la partition tolerance est obligatoire dans les réseaux. On choisit donc entre CP comme MongoDB ou AP comme Cassandra.",
        metadata={"category": "database", "author": "Jean Martin", "year": 2024},
    ),
    Document(
        page_content="Apache Cassandra est un column store distribué conçu pour la scalabilité linéaire. L'architecture peer-to-peer évite les single points of failure. Le consistent hashing distribue les données uniformément. Les niveaux de consistency permettent d'équilibrer performance et cohérence.",
        metadata={"category": "database", "author": "Sophie Chen", "year": 2023},
    ),
    Document(
        page_content="Elasticsearch est basé sur Apache Lucene pour le full-text search. Les documents JSON sont indexés dans des inverted indexes. Les analyzers décomposent le texte en tokens. Le mapping définit les types de champs et les analyzers à utiliser.",
        metadata={"category": "database", "author": "Pierre Lefebvre", "year": 2024},
    ),
    Document(
        page_content="Les graph databases comme Neo4j excellent pour les données hautement relationnelles. Les nodes représentent les entités et les relationships les connexions. Le langage Cypher permet des requêtes de traversée expressives.",
        metadata={"category": "database", "author": "Marie Dubois", "year": 2024},
    ),
    Document(
        page_content="InfluxDB est optimisé pour les time-series data. Les measurements stockent les points de données avec des tags et fields. Les tags sont indexés pour les filtres rapides. La retention policy supprime automatiquement les vieilles données.",
        metadata={"category": "database", "author": "Jean Martin", "year": 2023},
    ),
    Document(
        page_content="Le choix entre SQL et NoSQL dépend du use case. SQL convient aux données structurées avec relations complexes et transactions ACID. NoSQL excelle pour les données non-structurées, la scalabilité horizontale et les lectures massives.",
        metadata={"category": "database", "author": "Sophie Chen", "year": 2024},
    ),
    Document(
        page_content="La réplication copie les données entre serveurs pour la disponibilité. La réplication master-slave route les écritures vers le master et les lectures vers les slaves. La réplication multi-master permet les écritures sur plusieurs nodes mais nécessite la résolution de conflits.",
        metadata={"category": "database", "author": "Pierre Lefebvre", "year": 2024},
    ),
    Document(
        page_content="Les index accélèrent les recherches mais ralentissent les écritures. Les B-tree indexes sont standards pour les ranges. Les hash indexes optimisent les égalités exactes. Les bitmap indexes conviennent aux colonnes à faible cardinalité.",
        metadata={"category": "database", "author": "Marie Dubois", "year": 2023},
    ),
    Document(
        page_content="Les systèmes OLTP (Online Transaction Processing) gèrent de nombreuses transactions courtes. Les systèmes OLAP (Online Analytical Processing) exécutent des requêtes complexes sur de gros volumes. La dénormalisation et les column stores accélèrent les agrégations.",
        metadata={"category": "database", "author": "Jean Martin", "year": 2024},
    ),
    Document(
        page_content="Le sharding partitionne horizontalement les données pour la scalabilité. Le range sharding distribue selon des ranges de clés. Le hash sharding utilise une fonction de hash pour une distribution uniforme. Le geographic sharding localise les données près des utilisateurs.",
        metadata={"category": "database", "author": "Sophie Chen", "year": 2024},
    ),
    # Documents Machine Learning (20)
    Document(
        page_content="La régularisation prévient l'overfitting dans les réseaux de neurones. Le dropout désactive aléatoirement des neurones pendant l'entraînement, forçant le réseau à apprendre des représentations robustes. La régularisation L2 (weight decay) pénalise les poids élevés dans la fonction de coût. Le batch normalization normalise les activations entre les layers.",
        metadata={"category": "ml", "author": "Dr. Laurent Bernard", "year": 2024},
    ),
    Document(
        page_content="Le dropout est une technique de régularisation où des neurones sont aléatoirement supprimés pendant l'entraînement avec une probabilité p (typiquement 0.5). Cela empêche les co-adaptations complexes entre neurones. À l'inférence, tous les neurones sont utilisés mais leurs sorties sont multipliées par (1-p).",
        metadata={"category": "ml", "author": "Dr. Emma Wilson", "year": 2023},
    ),
    Document(
        page_content="Le batch normalization normalise les activations de chaque layer en utilisant la moyenne et variance du mini-batch. Cela réduit l'internal covariate shift et permet d'utiliser des learning rates plus élevés. Les paramètres gamma et beta sont appris pour permettre au réseau d'annuler la normalisation si nécessaire.",
        metadata={"category": "ml", "author": "Dr. Laurent Bernard", "year": 2024},
    ),
    Document(
        page_content="La régularisation L2 ajoute un terme lambda * sum(w²) à la loss function. Cela pénalise les poids élevés et encourage un modèle plus simple. Le weight decay implémente la même chose en soustrayant directement lambda*w des poids à chaque step.",
        metadata={"category": "ml", "author": "Dr. Emma Wilson", "year": 2024},
    ),
    Document(
        page_content="L'augmentation de données crée des variantes d'images pour enrichir le dataset. Les transformations incluent rotations, flips, crops, changements de luminosité/contraste. Le mixup combine linéairement deux images et leurs labels. Le cutout masque des rectangles aléatoires.",
        metadata={"category": "ml", "author": "Dr. Laurent Bernard", "year": 2023},
    ),
    Document(
        page_content="Le gradient clipping limite la magnitude des gradients pour éviter les explosions. Le clipping by value limite chaque élément du gradient. Le clipping by norm rescale le gradient si sa norme dépasse un seuil. C'est essentiel pour les RNNs avec backpropagation through time.",
        metadata={"category": "ml", "author": "Dr. Emma Wilson", "year": 2024},
    ),
    Document(
        page_content="Le learning rate schedule adapte le learning rate pendant l'entraînement. Le step decay divise le LR par un facteur tous les N epochs. L'exponential decay multiplie par gamma à chaque epoch. Le cosine annealing suit une courbe cosinus décroissante.",
        metadata={"category": "ml", "author": "Dr. Laurent Bernard", "year": 2024},
    ),
    Document(
        page_content="Adam combine momentum et RMSprop en maintenant des moyennes exponentielles des gradients et leurs carrés. Les hyperparamètres beta1 (0.9) et beta2 (0.999) contrôlent les decay rates. AdamW découple le weight decay de l'optimisation pour de meilleures performances.",
        metadata={"category": "ml", "author": "Dr. Emma Wilson", "year": 2023},
    ),
    Document(
        page_content="Le transfer learning réutilise des modèles pré-entraînés sur de nouvelles tâches. On gèle généralement les premières couches qui extraient des features génériques et on fine-tune les dernières. Le learning rate pour les couches pré-entraînées doit être plus faible.",
        metadata={"category": "ml", "author": "Dr. Laurent Bernard", "year": 2024},
    ),
    Document(
        page_content="Les CNNs modernes utilisent des blocs répétitifs pour la profondeur. ResNet introduit les skip connections pour éviter le vanishing gradient. DenseNet connecte chaque layer à tous les suivants. EfficientNet scale uniformément depth, width et resolution.",
        metadata={"category": "ml", "author": "Dr. Emma Wilson", "year": 2024},
    ),
    Document(
        page_content="L'attention permet au modèle de se concentrer sur des parties pertinentes de l'input. Le self-attention calcule l'attention entre toutes les positions d'une séquence. Le scaled dot-product attention utilise queries, keys et values. Le multi-head attention applique plusieurs attention heads en parallèle.",
        metadata={"category": "ml", "author": "Dr. Laurent Bernard", "year": 2024},
    ),
    Document(
        page_content="BERT (Bidirectional Encoder Representations from Transformers) est pré-entraîné avec masked language modeling. Des tokens aléatoires sont masqués et le modèle doit les prédire. Le contexte bidirectionnel capture les dépendances gauche et droite.",
        metadata={"category": "ml", "author": "Dr. Emma Wilson", "year": 2023},
    ),
    Document(
        page_content="GPT (Generative Pre-trained Transformer) est un modèle de langage autorégressif. Il prédit le token suivant conditionnellement aux précédents. Le pré-entraînement sur de vastes corpus textuels apprend des représentations riches. Le prompting guide le modèle sans fine-tuning.",
        metadata={"category": "ml", "author": "Dr. Laurent Bernard", "year": 2024},
    ),
    Document(
        page_content="L'accuracy mesure le taux de prédictions correctes mais est trompeur avec des classes déséquilibrées. La precision mesure la proportion de positifs prédits qui sont vrais. Le recall mesure la proportion de vrais positifs détectés. Le F1-score est la moyenne harmonique de precision et recall.",
        metadata={"category": "ml", "author": "Dr. Emma Wilson", "year": 2024},
    ),
    Document(
        page_content="La validation croisée k-fold divise les données en k sous-ensembles. Le modèle est entraîné k fois, chaque fois avec un fold différent comme validation. Cela fournit une estimation robuste de la performance. Le stratified k-fold préserve la distribution des classes.",
        metadata={"category": "ml", "author": "Dr. Laurent Bernard", "year": 2023},
    ),
    Document(
        page_content="L'ensemble learning combine plusieurs modèles pour améliorer les performances. Le bagging entraîne des modèles indépendants sur des échantillons bootstrap et moyenne leurs prédictions. Le boosting entraîne séquentiellement des modèles sur les erreurs des précédents.",
        metadata={"category": "ml", "author": "Dr. Emma Wilson", "year": 2024},
    ),
    Document(
        page_content="XGBoost est une implémentation optimisée du gradient boosting. Il utilise une régularisation sur les feuilles des arbres. Le split finding approximatif accélère l'entraînement. Les hyperparamètres clés sont max_depth, learning_rate, et n_estimators.",
        metadata={"category": "ml", "author": "Dr. Laurent Bernard", "year": 2024},
    ),
    Document(
        page_content="Les GANs utilisent deux réseaux en compétition : un générateur et un discriminateur. Le générateur crée des échantillons synthétiques. Le discriminateur distingue vrais et faux échantillons. Le mode collapse fait que le générateur produit peu de variété.",
        metadata={"category": "ml", "author": "Dr. Emma Wilson", "year": 2023},
    ),
    Document(
        page_content="Le Q-learning apprend une fonction Q(state, action) estimant la récompense future. L'agent explore l'environnement et met à jour Q avec l'équation de Bellman. Le Deep Q-Network (DQN) utilise un réseau neuronal pour approximer Q.",
        metadata={"category": "ml", "author": "Dr. Laurent Bernard", "year": 2024},
    ),
    Document(
        page_content="Les autoencoders apprennent des représentations compressées non supervisées. L'encoder compresse l'input en une représentation latente. Le decoder reconstruit l'input original. Les variational autoencoders (VAE) apprennent des distributions latentes continues.",
        metadata={"category": "ml", "author": "Dr. Emma Wilson", "year": 2024},
    ),
    # Documents RAG (25)
    Document(
        page_content="Le Retrieval-Augmented Generation (RAG) combine recherche d'information et génération par LLM. Le pipeline comprend chunking des documents, embedding, indexation vectorielle, retrieval et génération. Le RAG réduit les hallucinations en ancrant les réponses dans des sources.",
        metadata={"category": "rag", "author": "Dr. Sarah Kim", "year": 2024},
    ),
    Document(
        page_content="Les embeddings transforment le texte en vecteurs denses capturant la sémantique. Les modèles sentence-transformers sont optimisés pour les tâches de similarité. Le modèle all-MiniLM-L6-v2 offre un bon compromis taille/performance.",
        metadata={"category": "rag", "author": "Dr. Michael Tran", "year": 2024},
    ),
    Document(
        page_content="Le chunking découpe les documents en morceaux pour l'indexation. Le fixed-size chunking utilise une taille fixe avec overlap. Le RecursiveCharacterTextSplitter respecte la structure. Le SemanticChunker utilise les embeddings pour détecter les ruptures sémantiques.",
        metadata={"category": "rag", "author": "Dr. Sarah Kim", "year": 2023},
    ),
    Document(
        page_content="Les vector stores indexent et recherchent efficacement les embeddings. ChromaDB est open-source et simple à utiliser localement. Pinecone offre une solution cloud managée avec faible latence. FAISS optimise les recherches à grande échelle.",
        metadata={"category": "rag", "author": "Dr. Michael Tran", "year": 2024},
    ),
    Document(
        page_content="La similarité cosinus mesure l'angle entre deux vecteurs, variant de -1 à 1. Elle ignore la magnitude et se concentre sur la direction. La formule est cos(θ) = (A·B) / (||A|| ||B||). Une similarité de 1 indique des vecteurs identiques.",
        metadata={"category": "rag", "author": "Dr. Sarah Kim", "year": 2024},
    ),
    Document(
        page_content="Le MMR équilibre pertinence et diversité des résultats. L'algorithme sélectionne itérativement des documents maximisant λ*relevance - (1-λ)*similarity_to_selected. Le paramètre λ contrôle le trade-off : λ=1 maximise la pertinence, λ=0 maximise la diversité.",
        metadata={"category": "rag", "author": "Dr. Michael Tran", "year": 2024},
    ),
    Document(
        page_content="Le hybrid search combine recherche par keywords (BM25) et sémantique (vecteurs). BM25 excelle sur les termes techniques exacts et acronymes. La recherche vectorielle capture la similarité sémantique. L'EnsembleRetriever fusionne les résultats avec des poids configurables.",
        metadata={"category": "rag", "author": "Dr. Sarah Kim", "year": 2023},
    ),
    Document(
        page_content="BM25 est un algorithme de ranking basé sur TF-IDF amélioré. Il utilise la saturation du term frequency pour éviter le biais vers les documents longs. Les paramètres k1 contrôlent la saturation. BM25 excelle sur les exact matches et noms propres.",
        metadata={"category": "rag", "author": "Dr. Michael Tran", "year": 2024},
    ),
    Document(
        page_content="Le reranking utilise des cross-encoders pour réordonner les documents récupérés. Les bi-encoders (retrieval) encodent query et documents séparément. Les cross-encoders (reranking) encodent ensemble pour un scoring plus précis. Le reranking améliore significativement la pertinence mais est plus lent.",
        metadata={"category": "rag", "author": "Dr. Sarah Kim", "year": 2024},
    ),
    Document(
        page_content="Les bi-encoders encodent query et documents indépendamment. Les embeddings peuvent être pré-calculés et indexés. Les cross-encoders encodent query et document ensemble. Ils utilisent l'attention croisée pour un scoring fin mais sont plus lents.",
        metadata={"category": "rag", "author": "Dr. Michael Tran", "year": 2023},
    ),
    Document(
        page_content="Le Self-Query Retriever extrait automatiquement les filtres de métadonnées depuis la query. Un LLM analyse la question et identifie les filtres structurés. Le self-query nécessite un schéma de métadonnées bien défini.",
        metadata={"category": "rag", "author": "Dr. Sarah Kim", "year": 2024},
    ),
    Document(
        page_content="Le Multi-Query Retriever génère plusieurs variantes de la question originale. Un LLM reformule la query sous différents angles. Chaque variante est recherchée indépendamment. Les résultats sont fusionnés pour enrichir le contexte.",
        metadata={"category": "rag", "author": "Dr. Michael Tran", "year": 2024},
    ),
    Document(
        page_content="Le Parent Document Retriever recherche via des petits chunks mais retourne les documents parents complets. Les petits chunks améliorent la précision de la recherche. Les documents parents fournissent plus de contexte au LLM.",
        metadata={"category": "rag", "author": "Dr. Sarah Kim", "year": 2023},
    ),
    Document(
        page_content="HyDE génère un document hypothétique répondant à la question, puis le recherche. Le LLM génère une réponse possible même sans contexte. L'embedding de cette réponse est souvent plus proche des documents pertinents.",
        metadata={"category": "rag", "author": "Dr. Michael Tran", "year": 2024},
    ),
    Document(
        page_content="Le prompt structure le contexte et la question pour optimiser les réponses. Le few-shot prompting fournit des exemples de réponses souhaitées. Les instructions explicites guident le comportement du LLM. Les citations ancrent les réponses dans le contexte.",
        metadata={"category": "rag", "author": "Dr. Sarah Kim", "year": 2024},
    ),
    Document(
        page_content="L'évaluation du RAG mesure retrieval et génération séparément. Le retrieval est évalué avec precision@k, recall@k, MRR. La génération est évaluée avec faithfulness et answer relevance. Le RAGAS framework fournit des métriques automatiques.",
        metadata={"category": "rag", "author": "Dr. Michael Tran", "year": 2024},
    ),
    Document(
        page_content="Le chunk overlap duplique du contenu aux frontières entre chunks. Un overlap de 10-20% préserve le contexte aux limites. Trop d'overlap gaspille de l'espace. Trop peu d'overlap risque de couper des informations critiques.",
        metadata={"category": "rag", "author": "Dr. Sarah Kim", "year": 2023},
    ),
    Document(
        page_content="Les métadonnées enrichissent les documents pour le filtrage et le routing. Les métadonnées typiques incluent source, date, auteur, catégorie, langue. Le filtering pré-recherche restreint aux documents pertinents.",
        metadata={"category": "rag", "author": "Dr. Michael Tran", "year": 2024},
    ),
    Document(
        page_content="FAISS (Facebook AI Similarity Search) optimise la recherche vectorielle sur des milliards de vecteurs. L'indexation IVF partitionne l'espace vectoriel. Le product quantization compresse les vecteurs pour réduire la mémoire.",
        metadata={"category": "rag", "author": "Dr. Sarah Kim", "year": 2024},
    ),
    Document(
        page_content="Pinecone est une base vectorielle cloud managée. Elle offre des latences sub-milliseconde à grande échelle. Les namespaces isolent les données par tenant. Les métadonnées permettent le filtrage hybride.",
        metadata={"category": "rag", "author": "Dr. Michael Tran", "year": 2023},
    ),
    Document(
        page_content="Weaviate combine recherche vectorielle et graphe de connaissances. Le schema définit les classes et leurs propriétés. Les cross-references lient les objets entre eux. Le hybrid search combine vecteurs et BM25 nativement.",
        metadata={"category": "rag", "author": "Dr. Sarah Kim", "year": 2024},
    ),
    Document(
        page_content="Le context window limite la quantité de texte fournie au LLM. GPT-3.5 a 4K tokens, GPT-4 jusqu'à 128K tokens. Le RAG doit sélectionner les chunks les plus pertinents. Les longs contextes augmentent la latence et le coût.",
        metadata={"category": "rag", "author": "Dr. Michael Tran", "year": 2024},
    ),
    Document(
        page_content="Le fine-tuning adapte les embeddings au domaine spécifique. Les paires query-document pertinentes servent de données d'entraînement. La contrastive loss rapproche les paires positives et éloigne les négatives.",
        metadata={"category": "rag", "author": "Dr. Sarah Kim", "year": 2023},
    ),
    Document(
        page_content="Le caching réduit les coûts et latences en réutilisant les résultats. Les embeddings peuvent être pré-calculés et mis en cache. Les queries fréquentes peuvent cacher les résultats complets. Le semantic caching matche les queries similaires.",
        metadata={"category": "rag", "author": "Dr. Michael Tran", "year": 2024},
    ),
    Document(
        page_content="Le RAG et le fine-tuning résolvent des problèmes différents. Le RAG injecte des connaissances actualisées sans réentraînement. Le fine-tuning modifie le comportement et le style du modèle. La combinaison RAG + fine-tuning offre le meilleur des deux mondes.",
        metadata={"category": "rag", "author": "Dr. Sarah Kim", "year": 2024},
    ),
    # Documents DevOps (25)
    Document(
        page_content="Docker permet d'empaqueter des applications avec leurs dépendances dans des conteneurs portables et légers. Un Dockerfile définit les étapes de construction de l'image. Les layers sont mis en cache pour accélérer les builds. Les volumes persistent les données hors du conteneur.",
        metadata={"category": "devops", "author": "Alex Johnson", "year": 2024},
    ),
    Document(
        page_content="Kubernetes orchestre des conteneurs à grande échelle avec auto-scaling et self-healing. Les Pods sont les plus petites unités déployables. Les Deployments gèrent les Pods et leur mise à jour. Les Services exposent les Pods avec load balancing.",
        metadata={"category": "devops", "author": "Maria Rodriguez", "year": 2024},
    ),
    Document(
        page_content="Les bonnes pratiques Dockerfile optimisent taille et sécurité. Utilisez des images de base minimales comme alpine. Ordonnez les instructions de la moins à la plus changeante pour maximiser le cache. Le multi-stage build sépare build et runtime.",
        metadata={"category": "devops", "author": "Alex Johnson", "year": 2023},
    ),
    Document(
        page_content="Les Deployments gèrent le cycle de vie des Pods. Le rolling update déploie progressivement les nouvelles versions. Le paramètre maxUnavailable contrôle le nombre de Pods indisponibles. Les health checks garantissent que les nouveaux Pods sont prêts.",
        metadata={"category": "devops", "author": "Maria Rodriguez", "year": 2024},
    ),
    Document(
        page_content="Docker Compose définit des applications multi-conteneurs en YAML. Les services décrivent les conteneurs à déployer. Les depends_on contrôlent l'ordre de démarrage. Les networks isolent les services. Les volumes partagent des données entre conteneurs.",
        metadata={"category": "devops", "author": "Alex Johnson", "year": 2024},
    ),
    Document(
        page_content="Les Services Kubernetes exposent les Pods avec load balancing stable. Le ClusterIP expose en interne uniquement. Le NodePort ouvre un port sur tous les nodes. Le LoadBalancer provisionne un load balancer cloud.",
        metadata={"category": "devops", "author": "Maria Rodriguez", "year": 2023},
    ),
    Document(
        page_content="L'intégration continue (CI) automatise les builds et tests. Le déploiement continu (CD) automatise les déploiements. GitLab CI utilise des pipelines définis en .gitlab-ci.yml. Les stages structurent le pipeline (build, test, deploy).",
        metadata={"category": "devops", "author": "Alex Johnson", "year": 2024},
    ),
    Document(
        page_content="Les ConfigMaps stockent la configuration non-sensible. Les Secrets stockent les données sensibles encodées en base64. Ils peuvent être montés comme volumes ou exposés comme environment variables. Le RBAC contrôle l'accès aux Secrets.",
        metadata={"category": "devops", "author": "Maria Rodriguez", "year": 2024},
    ),
    Document(
        page_content="Terraform gère l'infrastructure cloud de manière déclarative. Les providers interagissent avec les APIs cloud. Les ressources définissent l'infrastructure à créer. Les variables paramètrent les configurations. Les modules réutilisent des configurations.",
        metadata={"category": "devops", "author": "Alex Johnson", "year": 2023},
    ),
    Document(
        page_content="L'Ingress expose les services HTTP/HTTPS hors du cluster. Il fournit load balancing, SSL termination et name-based virtual hosting. Un Ingress Controller implémente les règles Ingress. Nginx et Traefik sont des controllers populaires.",
        metadata={"category": "devops", "author": "Maria Rodriguez", "year": 2024},
    ),
    Document(
        page_content="Prometheus collecte et stocke des métriques time-series. Le scraping récupère les métriques exposées par les applications. Le PromQL interroge les métriques. Grafana visualise les métriques avec des dashboards.",
        metadata={"category": "devops", "author": "Alex Johnson", "year": 2024},
    ),
    Document(
        page_content="Les StatefulSets gèrent les applications stateful nécessitant une identité stable. Chaque Pod a un nom prévisible et un PersistentVolume dédié. Le déploiement et scaling sont ordonnés. Les StatefulSets sont idéaux pour les bases de données.",
        metadata={"category": "devops", "author": "Maria Rodriguez", "year": 2023},
    ),
    Document(
        page_content="GitOps utilise Git comme source de vérité pour l'infrastructure. Les changements passent par des pull requests. ArgoCD synchronise automatiquement le cluster avec Git. Il détecte les drifts entre Git et le cluster.",
        metadata={"category": "devops", "author": "Alex Johnson", "year": 2024},
    ),
    Document(
        page_content="Un service mesh gère la communication entre microservices. Istio injecte des sidecars Envoy proxy dans chaque Pod. Le traffic management route et load balance les requêtes. Le mutual TLS sécurise la communication.",
        metadata={"category": "devops", "author": "Maria Rodriguez", "year": 2024},
    ),
    Document(
        page_content="Le HPA scale automatiquement le nombre de Pods selon les métriques. Le CPU et la mémoire sont des métriques standard. Les custom metrics utilisent des métriques applicatives. Les limits min et max contraignent le scaling.",
        metadata={"category": "devops", "author": "Alex Johnson", "year": 2023},
    ),
    Document(
        page_content="Ansible automatise la configuration et le déploiement de manière agentless. Les playbooks décrivent l'état désiré en YAML. Les modules exécutent des tâches spécifiques. L'inventaire liste les hosts et groupes. Les roles organisent les playbooks réutilisables.",
        metadata={"category": "devops", "author": "Maria Rodriguez", "year": 2024},
    ),
    Document(
        page_content="Les canary releases déploient progressivement à un subset d'utilisateurs. Le trafic est routé graduellement vers la nouvelle version. Les métriques sont monitorées pour détecter les régressions. Le rollback automatique annule si les métriques se dégradent.",
        metadata={"category": "devops", "author": "Alex Johnson", "year": 2024},
    ),
    Document(
        page_content="Les feature flags découplent le déploiement de la release. Les nouvelles features sont déployées désactivées. Les flags activent les features progressivement. Les kill switches désactivent rapidement une feature problématique.",
        metadata={"category": "devops", "author": "Maria Rodriguez", "year": 2023},
    ),
    Document(
        page_content="ELK (Elasticsearch, Logstash, Kibana) agrège et analyse les logs. Logstash ingère et transforme les logs. Elasticsearch indexe et stocke les logs. Kibana visualise et recherche les logs.",
        metadata={"category": "devops", "author": "Alex Johnson", "year": 2024},
    ),
    Document(
        page_content="Les health checks vérifient l'état des services. Le liveness probe redémarre les conteneurs défaillants. Le readiness probe retire les conteneurs non-prêts du load balancing. Le startup probe donne du temps aux applications lentes à démarrer.",
        metadata={"category": "devops", "author": "Maria Rodriguez", "year": 2024},
    ),
    Document(
        page_content="Les namespaces isolent les ressources dans un cluster. Ils permettent le multi-tenancy logique. Les ResourceQuotas limitent les ressources par namespace. Le RBAC contrôle l'accès aux namespaces. Les NetworkPolicies isolent le trafic réseau.",
        metadata={"category": "devops", "author": "Alex Johnson", "year": 2023},
    ),
    Document(
        page_content="Vault sécurise, stocke et contrôle l'accès aux secrets. Le dynamic secrets génère des credentials à la demande. L'encryption as a service chiffre les données. Les auth methods authentifient les clients. Les policies contrôlent l'accès aux secrets.",
        metadata={"category": "devops", "author": "Maria Rodriguez", "year": 2024},
    ),
    Document(
        page_content="Le chaos engineering teste la résilience en injectant des pannes. Il identifie les faiblesses avant qu'elles causent des incidents. Chaos Monkey de Netflix termine aléatoirement des instances. Litmus chaos effectue des chaos experiments sur Kubernetes.",
        metadata={"category": "devops", "author": "Alex Johnson", "year": 2024},
    ),
    Document(
        page_content="Les SLIs (Service Level Indicators) mesurent la performance. Les SLOs (Service Level Objectives) définissent les cibles acceptables. L'error budget est la différence entre 100% et le SLO. Il permet d'équilibrer innovation et fiabilité.",
        metadata={"category": "devops", "author": "Maria Rodriguez", "year": 2023},
    ),
    Document(
        page_content="Le serverless exécute du code sans gérer l'infrastructure. AWS Lambda, Azure Functions et Google Cloud Functions sont des FaaS. Le cold start est la latence au premier invocation. Le pricing est basé sur les invocations et durée.",
        metadata={"category": "devops", "author": "Alex Johnson", "year": 2024},
    ),
    # Documents Python (15)
    Document(
        page_content="Asyncio permet la programmation asynchrone en Python avec async/await. Les coroutines sont des fonctions définies avec async def. Le keyword await suspend l'exécution jusqu'à ce que la coroutine soit complète. L'event loop schedule et exécute les coroutines.",
        metadata={"category": "python", "author": "Thomas Dupont", "year": 2024},
    ),
    Document(
        page_content="Les type hints annotent les types des variables et fonctions. Ils améliorent la lisibilité et permettent la vérification statique. MyPy est un type checker qui détecte les erreurs de types. Les generic types comme List[int] spécifient les types contenus.",
        metadata={"category": "python", "author": "Claire Martin", "year": 2024},
    ),
    Document(
        page_content="Les decorators modifient le comportement des fonctions et classes. Un decorator est une fonction qui prend une fonction et retourne une fonction modifiée. La syntaxe @decorator s'applique au-dessus de la définition.",
        metadata={"category": "python", "author": "Thomas Dupont", "year": 2023},
    ),
    Document(
        page_content="Les context managers gèrent l'acquisition et libération de ressources. Le statement with garantit la cleanup même en cas d'exception. Les méthodes __enter__ et __exit__ définissent un context manager.",
        metadata={"category": "python", "author": "Claire Martin", "year": 2024},
    ),
    Document(
        page_content="Les generators produisent des valeurs à la demande avec yield. Ils économisent la mémoire en ne calculant qu'une valeur à la fois. Les generator expressions (x for x in range(10)) sont compactes. itertools fournit des combinators pour generators.",
        metadata={"category": "python", "author": "Thomas Dupont", "year": 2024},
    ),
    Document(
        page_content="Les metaclasses contrôlent la création des classes. type est la metaclasse par défaut. Les metaclasses customisées héritent de type. La méthode __new__ crée la classe. Les metaclasses permettent de valider les attributs de classe.",
        metadata={"category": "python", "author": "Claire Martin", "year": 2023},
    ),
    Document(
        page_content="Les dataclasses simplifient la création de classes pour stocker des données. Le decorator @dataclass génère automatiquement __init__, __repr__ et __eq__. Les field() permet de customiser les champs avec des default values.",
        metadata={"category": "python", "author": "Thomas Dupont", "year": 2024},
    ),
    Document(
        page_content="Pathlib fournit une approche orientée objet pour les chemins de fichiers. Les objets Path représentent les chemins. L'opérateur / concatène les chemins. Les méthodes exists(), is_file(), is_dir() testent l'existence.",
        metadata={"category": "python", "author": "Claire Martin", "year": 2024},
    ),
    Document(
        page_content="Les Enums définissent des ensembles de constantes nommées. Ils héritent de enum.Enum. Les membres sont accessibles par nom et valeur. Les Enums sont immutables et itérables. Le decorator @unique garantit que les valeurs sont uniques.",
        metadata={"category": "python", "author": "Thomas Dupont", "year": 2023},
    ),
    Document(
        page_content="Functools fournit des outils pour la programmation fonctionnelle. Le partial fixe certains arguments d'une fonction. Le reduce applique une fonction cumulativement à une séquence. Le lru_cache met en cache les résultats de fonctions.",
        metadata={"category": "python", "author": "Claire Martin", "year": 2024},
    ),
    Document(
        page_content="La sérialisation convertit les objets en bytes pour le stockage ou transport. Pickle sérialise les objets Python natifs. JSON sérialise en format texte interopérable. Protocol Buffers (protobuf) est un format binaire compact de Google.",
        metadata={"category": "python", "author": "Thomas Dupont", "year": 2024},
    ),
    Document(
        page_content="Le multiprocessing contourne le GIL en utilisant des processes séparés. Chaque process a son propre interpréteur Python et mémoire. Le Pool exécute des fonctions sur plusieurs workers. Les Queue partagent des données entre processes.",
        metadata={"category": "python", "author": "Claire Martin", "year": 2023},
    ),
    Document(
        page_content='Les f-strings offrent un formatage de chaînes concis et lisible. La syntaxe f"{variable}" interpole les variables. Les expressions {2+2} sont évaluées. Le format spec {value:.2f} contrôle le formatage. Les f-strings sont plus rapides que format().',
        metadata={"category": "python", "author": "Thomas Dupont", "year": 2024},
    ),
    Document(
        page_content="Les annotations attachent des métadonnées aux fonctions et variables. Elles sont stockées dans __annotations__. Les frameworks comme FastAPI et Pydantic exploitent les annotations runtime. Le module inspect accède aux annotations.",
        metadata={"category": "python", "author": "Claire Martin", "year": 2024},
    ),
    Document(
        page_content="Le pattern matching (match-case) introduit en Python 3.10 améliore les conditions complexes. Les patterns peuvent matcher des valeurs littérales, structures et types. Le wildcard _ matche n'importe quoi. Les guards if ajoutent des conditions.",
        metadata={"category": "python", "author": "Thomas Dupont", "year": 2023},
    ),
]

print(f"✓ Dataset créé : {len(documents)} documents")
print(f"✓ Catégories : {set(d.metadata['category'] for d in documents)}")
print(
    f"✓ Auteurs : {len(set(d.metadata['author'] for d in documents))} auteurs uniques"
)


✓ Dataset créé : 100 documents
✓ Catégories : {'python', 'database', 'rag', 'devops', 'ml'}
✓ Auteurs : 12 auteurs uniques


### TODO 2: Indexer dans ChromaDB

In [3]:
# TODO: Creer l'index vectoriel
embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")

vectorstore = Chroma.from_documents(
    documents=documents, embedding=embeddings, collection_name="tech_articles"
)

C:\Users\Administrateur\AppData\Local\Temp\ipykernel_3104\2768893370.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10122.94it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### TODO 3: Implementer Similarity Search

In [4]:
# TODO: Tester similarity_search sur les 5 questions
queries = [
    "Comment optimiser les performances d'une base de donnees PostgreSQL ?",
    "Quelles sont les techniques de regularisation en deep learning ?",
    "Docker Kubernetes orchestration conteneurs",
    "Expliquer le reranking dans un systeme RAG",
    "API REST OAuth authentication security",
]

for query in queries:
    results = vectorstore.similarity_search_with_score(query, k=5)
    print(query)
    for i, (doc, score) in enumerate(results, 1):
        print(f"{i} | Score : {score:<5.2f} | content : {doc.page_content[:100]}")
    print()

Comment optimiser les performances d'une base de donnees PostgreSQL ?
1 | Score : 6.19  | content : En production, PostgreSQL nécessite une configuration minutieuse. Le paramètre max_connections doit 
2 | Score : 7.20  | content : Pour optimiser les performances d'une base de données PostgreSQL, plusieurs techniques sont essentie
3 | Score : 12.13 | content : Le choix entre SQL et NoSQL dépend du use case. SQL convient aux données structurées avec relations 
4 | Score : 12.37 | content : Le fine-tuning adapte les embeddings au domaine spécifique. Les paires query-document pertinentes se
5 | Score : 12.58 | content : InfluxDB est optimisé pour les time-series data. Les measurements stockent les points de données ave

Quelles sont les techniques de regularisation en deep learning ?
1 | Score : 13.28 | content : Le batch normalization normalise les activations de chaque layer en utilisant la moyenne et variance
2 | Score : 14.96 | content : XGBoost est une implémentation optimisée du grad

### TODO 4: Implementer MMR avec differents lambda

In [5]:
# TODO: Tester max_marginal_relevance_search avec lambda=0.3, 0.5, 0.7
lambda_values = [0.3, 0.5, 0.7]
query = queries[0]

for lambda_val in lambda_values:
    print(f"MMR : {lambda_val}")
    results = vectorstore.max_marginal_relevance_search(
        query,
        k=5,
        fetch_k=20,
        lambda_mult=lambda_val,
    )
    print(f"  {query}")
    for i, doc in enumerate(results, 1):
        print(f"  {i} | content : {doc.page_content[:100]}")
    print()

MMR : 0.3
  Comment optimiser les performances d'une base de donnees PostgreSQL ?
  1 | content : En production, PostgreSQL nécessite une configuration minutieuse. Le paramètre max_connections doit 
  2 | content : Le choix entre SQL et NoSQL dépend du use case. SQL convient aux données structurées avec relations 
  3 | content : L'ensemble learning combine plusieurs modèles pour améliorer les performances. Le bagging entraîne d
  4 | content : Le MMR équilibre pertinence et diversité des résultats. L'algorithme sélectionne itérativement des d
  5 | content : Apache Cassandra est un column store distribué conçu pour la scalabilité linéaire. L'architecture pe

MMR : 0.5
  Comment optimiser les performances d'une base de donnees PostgreSQL ?
  1 | content : En production, PostgreSQL nécessite une configuration minutieuse. Le paramètre max_connections doit 
  2 | content : Pour optimiser les performances d'une base de données PostgreSQL, plusieurs techniques sont essentie
  3 | content : 

### TODO 5: Implementer Hybrid Search (BM25 + Vectoriel)

In [6]:
# TODO: Utiliser EnsembleRetriever pour combiner BM25 et recherche vectorielle
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 5

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever], weights=[0.4, 0.6]
)

query = queries[0]
hybrid_results = ensemble_retriever.invoke(query)

print(query)
for i, doc in enumerate(hybrid_results, 1):
    print(f"{i:<2} | content : {doc.page_content[:100]}")

Comment optimiser les performances d'une base de donnees PostgreSQL ?
1  | content : Pour optimiser les performances d'une base de données PostgreSQL, plusieurs techniques sont essentie
2  | content : En production, PostgreSQL nécessite une configuration minutieuse. Le paramètre max_connections doit 
3  | content : Le choix entre SQL et NoSQL dépend du use case. SQL convient aux données structurées avec relations 
4  | content : Le fine-tuning adapte les embeddings au domaine spécifique. Les paires query-document pertinentes se
5  | content : InfluxDB est optimisé pour les time-series data. Les measurements stockent les points de données ave
6  | content : Le prompt structure le contexte et la question pour optimiser les réponses. Le few-shot prompting fo
7  | content : Les bonnes pratiques Dockerfile optimisent taille et sécurité. Utilisez des images de base minimales
8  | content : Functools fournit des outils pour la programmation fonctionnelle. Le partial fixe certains arguments
9 

### TODO 6: Analyser la diversite des resultats

In [7]:
# TODO: Pour chaque methode, calculer le nombre de categories/auteurs uniques dans les resultats
analysis = {
    "similarity_search": {"categories": set(), "authors": set()},
    "mmr_search": {"categories": set(), "authors": set()},
    "hybrid_search": {"categories": set(), "authors": set()},
}


def extract_meta(doc):
    """Adapte selon ton schéma de metadata."""
    meta = doc.metadata if hasattr(doc, "metadata") else {}
    category = meta.get("category")
    author = meta.get("author")
    return category, author


for query in queries:
    similarity_results = vectorstore.similarity_search_with_score(query, k=5)
    mmr_results = vectorstore.max_marginal_relevance_search(
        query,
        k=5,
        fetch_k=20,
        lambda_mult=0.5,
    )
    hybrid_results = ensemble_retriever.invoke(query)

    # similarity
    for doc, _ in similarity_results:
        cat, auth = extract_meta(doc)
        if cat:
            analysis["similarity_search"]["categories"].add(cat)
        if auth:
            analysis["similarity_search"]["authors"].add(auth)

    # MMR
    for doc in mmr_results:
        cat, auth = extract_meta(doc)
        if cat:
            analysis["mmr_search"]["categories"].add(cat)
        if auth:
            analysis["mmr_search"]["authors"].add(auth)

    # hybrid
    for doc in hybrid_results:
        cat, auth = extract_meta(doc)
        if cat:
            analysis["hybrid_search"]["categories"].add(cat)
        if auth:
            analysis["hybrid_search"]["authors"].add(auth)


# conversion finale en métriques exploitables
analysis_df = pd.DataFrame(
    {
        k: {
            "unique_categories": len(v["categories"]),
            "unique_authors": len(v["authors"]),
        }
        for k, v in analysis.items()
    }
).T

analysis_df.head()

,unique_categories,unique_authors
similarity_search,5,11
mmr_search,5,11
hybrid_search,5,12


### TODO 7: Benchmarker les performances

In [8]:
# TODO: Mesurer le temps de reponse pour chaque methode
latency_metrics = {
    "similarity_search": [],
    "mmr_search": [],
    "hybrid_search": [],
}


def run_benchmark(
    queries, vectorstore, ensemble_retriever, k=5, fetch_k=20, lambda_mult=0.5
):
    for query in queries:
        # Similarity search
        start = time.perf_counter()
        vectorstore.similarity_search_with_score(query, k=k)
        latency_metrics["similarity_search"].append(time.perf_counter() - start)

        # MMR search
        start = time.perf_counter()
        vectorstore.max_marginal_relevance_search(
            query,
            k=k,
            fetch_k=fetch_k,
            lambda_mult=lambda_mult,
        )
        latency_metrics["mmr_search"].append(time.perf_counter() - start)

        # Hybrid search
        start = time.perf_counter()
        ensemble_retriever.invoke(query)
        latency_metrics["hybrid_search"].append(time.perf_counter() - start)


run_benchmark(queries, vectorstore, ensemble_retriever)

# Résumé des performances
benchmark_summary = pd.DataFrame(
    {
        method: {
            "avg_latency_sec": np.mean(times),
            "median_latency_sec": np.median(times),
            "min_latency_sec": np.min(times),
            "max_latency_sec": np.max(times),
            "num_queries": len(times),
        }
        for method, times in latency_metrics.items()
    }
).T

benchmark_summary.head()

,avg_latency_sec,median_latency_sec,min_latency_sec,max_latency_sec,num_queries
similarity_search,0.009497,0.009602,0.007828,0.011723,5.0
mmr_search,0.012767,0.012813,0.010508,0.014378,5.0
hybrid_search,0.008963,0.008822,0.007947,0.010569,5.0
